# ICA-Based Second-Pass Pruning Scan for LLMs

## Hypothesis

A pretrained or already-pruned LLM can still contain residual weight redundancy that is not exposed by magnitude pruning, SVD/low-rank approximation, Wanda, or SparseGPT. These methods are highly effective, but they mostly rely on local weight magnitude, local activation magnitude, orthogonal low-rank structure, or local reconstruction criteria.

The working hypothesis is:

> If statistically separable activation components can be estimated inside a layer using ICA, then those components can be projected back into a **weight-level protection map**. This map can indicate which weights should be protected and which weights are plausible candidates for additional pruning.

A key point is that ICA is stricter than simply looking for an orthogonal basis. PCA/SVD finds mutually orthogonal directions that explain variance or matrix energy. ICA instead searches for components whose activations are as statistically independent as possible. The practical gain from this stricter criterion is a more selective protection signal: a component is not only a high-energy direction, but a direction with separable non-Gaussian usage structure.

This notebook does **not** use SAEs, sparse autoencoders, dictionary learning, or concept-discovery tooling. It does not train a new model. The goal is a minimal, executable research prototype for **ICA-based second-pass pruning diagnostics**.


## Core idea and formulas

Consider a linear module:

$$
y = W x
$$

where:

- $x \in \mathbb{R}^{d_{\mathrm{in}}}$ is the module input activation,
- $y \in \mathbb{R}^{d_{\mathrm{out}}}$ is the module output activation,
- $W \in \mathbb{R}^{d_{\mathrm{out}} \times d_{\mathrm{in}}}$ is the weight matrix.

### Orthogonal basis search versus ICA

An orthogonal basis method such as PCA/SVD searches for directions $q_k$ that satisfy:

$$
q_k^\top q_l = 0 \quad \text{for } k \ne l
$$

and then ranks those directions by variance or matrix energy. This is useful, but it is not the same as finding statistically separated activation sources.

ICA imposes a stricter condition. It tries to estimate latent sources $S_1, \ldots, S_K$ whose joint distribution factorizes as much as possible:

$$
p(S_1, \ldots, S_K)
\approx
\prod_{k=1}^{K} p(S_k)
$$

Equivalently, ICA tries to reduce the mutual information among components:

$$
I(S_1, \ldots, S_K)
=
\sum_{k=1}^{K} H(S_k)
-
H(S_1, \ldots, S_K)
$$

where $H(\cdot)$ denotes entropy.

In practice, FastICA first whitens the data and then rotates the whitened space toward non-Gaussian components. The stricter criterion is useful because it can separate activation sources that are not simply the largest orthogonal variance directions.

### ICA activation model

For a matrix of collected activation vectors $X$, ICA estimates:

$$
X \approx S A^\top
$$

where:

- $X \in \mathbb{R}^{n \times d}$ contains observed activation vectors ($n$ samples),
- $S \in \mathbb{R}^{n \times K}$ contains estimated source activations,
- $A \in \mathbb{R}^{d \times K}$ is the **mixing matrix** (column $k$ is the mixing direction of component $k$).

**Implementation note.** `sklearn.decomposition.FastICA` exposes `mixing_` (shape $[d, K]$, the mixing matrix $A$) and `components_` (shape $[K, d]$, the unmixing matrix $W$, where $S = X W^\top$). This prototype uses the **mixing matrix** to compute protection scores, so that the score reflects how strongly a dimension *participates* in each component, not how useful it is for *extracting* it.

### Component importance

As a first-order proxy, the importance of component $k$ is estimated by the mean absolute source activation:

$$
\alpha_k
=
\mathbb{E}\left[\left|S_k\right|\right]
$$

The values are normalized so that they sum to one:

$$
\tilde{\alpha}_k
=
\frac{\alpha_k}{\sum_{r=1}^{K} \alpha_r + \varepsilon}
$$

### Input-side protection

Let $a^{\mathrm{in}}_{k,j}$ be entry $j$ of the input-side mixing direction of component $k$ (i.e., row $k$ of `ica.mixing_.T @ pca.components_`).

For input dimension $j$, the protection score is:

$$
p^{\mathrm{in}}_j
=
\sum_{k=1}^{K}
\tilde{\alpha}^{\mathrm{in}}_k
\left|a^{\mathrm{in}}_{k,j}\right|
$$

### Output-side protection

Let $a^{\mathrm{out}}_{k,i}$ be entry $i$ of the output-side mixing direction of component $k$.

For output dimension $i$, the protection score is:

$$
p^{\mathrm{out}}_i
=
\sum_{k=1}^{K}
\tilde{\alpha}^{\mathrm{out}}_k
\left|a^{\mathrm{out}}_{k,i}\right|
$$

### Weight-level pruning score

Input-only mode:

$$
\mathrm{score}_{ij}
=
\left|W_{ij}\right| \cdot p^{\mathrm{in}}_j
$$

Input-output mode:

$$
\mathrm{score}_{ij}
=
\left|W_{ij}\right|
\cdot p^{\mathrm{out}}_i
\cdot p^{\mathrm{in}}_j
$$

A **low score** means that the weight is small and appears to be connected to weakly protected ICA directions. These weights become candidates for additional pruning.

What the stricter ICA criterion buys us: weights connected to statistically separable activation sources can be protected even when their raw magnitude, singular-vector energy, or channel activation norm is not large. This is the kind of residual structure that may remain after Wanda, SparseGPT, or SVD have already removed the obvious redundancy.



## 0. Colab setup

For Colab execution, select a GPU runtime before running the notebook:

**Runtime → Change runtime type → T4 GPU**

Small initial test models:

- `facebook/opt-125m`
- `gpt2`

Hooked activations can consume substantial memory on larger models. Start with conservative values for `num_samples`, `seq_len`, and `max_vectors_per_module`.


In [ ]:

!pip -q install transformers datasets accelerate scikit-learn pandas tqdm safetensors


In [ ]:

import os, json, math, gc, random, warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from sklearn.decomposition import FastICA, PCA
from sklearn.exceptions import ConvergenceWarning


## 1. Configuration

`target_name_keywords` controls which linear modules are scanned. For Llama/Mistral/OPT/GPT-style models, the following names are usually relevant:

- attention: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `c_attn`, `c_proj`
- MLP: `gate_proj`, `up_proj`, `down_proj`, `fc1`, `fc2`, `c_fc`


In [ ]:

VALID_DTYPES = {"float16", "bfloat16", "float32"}

@dataclass
class ScanConfig:
    model_name_or_path: str = "facebook/opt-125m"
    dataset_name: str = "wikitext"
    dataset_config: str = "wikitext-2-raw-v1"
    text_field: str = "text"
    num_samples: int = 32
    seq_len: int = 256
    n_components: int = 32
    extra_sparsity: float = 0.05
    max_vectors_per_module: int = 20000
    use_output_ica: bool = True
    ica_max_iter: int = 2000
    ica_tol: float = 1e-3
    skip_nonconverged_ica: bool = True
    apply_mask: bool = False
    output_dir: str = "outputs/ica_prune_scan"
    dtype: str = "float16"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    # FIX: reproducibility – fixed seed for calibration text shuffling
    random_seed: int = 42
    # FIX: batched activation collection (speed)
    collection_batch_size: int = 4
    target_name_keywords: Tuple[str, ...] = (
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        "fc1", "fc2", "c_fc", "c_proj", "c_attn"
    )

    def __post_init__(self):
        # FIX: validate dtype early instead of silently falling back to float32
        if self.dtype not in VALID_DTYPES:
            raise ValueError(
                f"Invalid dtype '{self.dtype}'. Choose from {VALID_DTYPES}."
            )

cfg = ScanConfig()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
print(cfg)


## 2. Load model and tokenizer

The model is run in `eval()` mode. This notebook performs no fine-tuning and no gradient-based updates.


In [ ]:

def get_torch_dtype(dtype_name: str):
    if dtype_name == "float16":
        return torch.float16
    if dtype_name == "bfloat16":
        return torch.bfloat16
    return torch.float32

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name_or_path,
    torch_dtype=get_torch_dtype(cfg.dtype),
    device_map="auto" if cfg.device == "cuda" else None,
)
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name_or_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()
print("Loaded:", cfg.model_name_or_path)


## 3. Calibration data

The calibration data is used to run the model and collect representative activations. The goal is not benchmarking; the goal is to observe typical activation patterns inside the selected modules.


In [ ]:

def load_calibration_texts(cfg: ScanConfig) -> List[str]:
    ds = load_dataset(cfg.dataset_name, cfg.dataset_config, split="train")
    texts = []
    for item in ds:
        t = item.get(cfg.text_field, "")
        if isinstance(t, str) and len(t.strip()) > 50:
            texts.append(t.strip())
        if len(texts) >= cfg.num_samples * 3:
            break
    # FIX: seeded shuffle for reproducibility
    rng = random.Random(cfg.random_seed)
    rng.shuffle(texts)
    return texts[:cfg.num_samples]


def load_eval_texts(cfg: ScanConfig, n_eval: int = 32) -> List[str]:
    """Load a separate held-out validation split for perplexity evaluation.
    Falls back to a different portion of the training split if 'validation'
    is not available for the chosen dataset.
    """
    try:
        ds = load_dataset(cfg.dataset_name, cfg.dataset_config, split="validation")
    except Exception:
        # Fallback: take a different tail slice from train with a different seed
        ds = load_dataset(cfg.dataset_name, cfg.dataset_config, split="train")
    texts = []
    for item in ds:
        t = item.get(cfg.text_field, "")
        if isinstance(t, str) and len(t.strip()) > 50:
            texts.append(t.strip())
        if len(texts) >= n_eval * 3:
            break
    rng = random.Random(cfg.random_seed + 1)  # different seed from calibration
    rng.shuffle(texts)
    return texts[:n_eval]


texts = load_calibration_texts(cfg)
eval_texts = load_eval_texts(cfg)
print("Calibration texts:", len(texts))
print("Eval texts:       ", len(eval_texts))
print(texts[0][:300])


In [ ]:

def tokenize_texts(texts: List[str], cfg: ScanConfig):
    enc = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=cfg.seq_len,
        return_tensors="pt"
    )
    return {k: v.to(model.device) for k, v in enc.items()}

batch = tokenize_texts(texts[:2], cfg)
print({k: tuple(v.shape) for k, v in batch.items()})


## 4. Select target linear modules

The notebook automatically finds `torch.nn.Linear` modules whose names contain at least one configured keyword. Embeddings and `lm_head` are excluded by default.


In [ ]:

def find_target_linears(model: nn.Module, cfg: ScanConfig) -> Dict[str, nn.Linear]:
    targets = {}
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        if "lm_head" in name or "embed" in name:
            continue
        if any(key in name for key in cfg.target_name_keywords):
            targets[name] = module
    return targets

target_modules = find_target_linears(model, cfg)
print("Target Linear modules:", len(target_modules))
for i, name in enumerate(target_modules.keys()):
    if i < 30:
        print(" ", name)
print("...")


## 5. Collect activations with forward hooks

For each targeted linear module, the notebook collects:

- input activations: \(X_{in}\)
- output activations: \(X_{out}\)

Token positions are subsampled to avoid excessive memory use. Activations are stored as CPU `float32` tensors.


In [ ]:

class ActivationCollector:
    def __init__(self, target_modules: Dict[str, nn.Module], max_vectors_per_module: int = 20000):
        self.target_modules = target_modules
        self.max_vectors = max_vectors_per_module
        self.inputs = {name: [] for name in target_modules}
        self.outputs = {name: [] for name in target_modules}
        self.handles = []

    def _sample_vectors(self, x: torch.Tensor, max_new: int = 512) -> torch.Tensor:
        if x.dim() >= 3:
            x = x.reshape(-1, x.shape[-1])
        n = x.shape[0]
        if n > max_new:
            idx = torch.randperm(n, device=x.device)[:max_new]
            x = x[idx]
        return x.detach().float().cpu()

    def _current_count(self, store: Dict[str, List[torch.Tensor]], name: str) -> int:
        return sum(t.shape[0] for t in store[name])

    def make_hook(self, name):
        def hook(module, inputs, output):
            if self._current_count(self.inputs, name) < self.max_vectors:
                self.inputs[name].append(self._sample_vectors(inputs[0]))
            if self._current_count(self.outputs, name) < self.max_vectors:
                self.outputs[name].append(self._sample_vectors(output))
        return hook

    def register(self):
        for name, module in self.target_modules.items():
            self.handles.append(module.register_forward_hook(self.make_hook(name)))

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles = []

    def finalize(self):
        X_in, X_out = {}, {}
        for name in self.target_modules:
            if self.inputs[name]:
                x = torch.cat(self.inputs[name], dim=0)[:self.max_vectors]
                X_in[name] = x.numpy()
            if self.outputs[name]:
                y = torch.cat(self.outputs[name], dim=0)[:self.max_vectors]
                X_out[name] = y.numpy()
        return X_in, X_out


In [ ]:

collector = ActivationCollector(
    target_modules,
    max_vectors_per_module=cfg.max_vectors_per_module,
)
collector.register()

# FIX: try/finally guarantees hooks are removed even if an exception occurs
try:
    with torch.no_grad():
        # FIX: batched forward passes for speed
        for i in tqdm(
            range(0, len(texts), cfg.collection_batch_size),
            desc="Collecting activations",
        ):
            batch_texts = texts[i : i + cfg.collection_batch_size]
            batch = tokenize_texts(batch_texts, cfg)
            _ = model(**batch)
finally:
    collector.remove()

X_in, X_out = collector.finalize()

print("Collected modules:", len(X_in))
first_name = next(iter(X_in))
print(first_name, "X_in", X_in[first_name].shape, "X_out", X_out[first_name].shape)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()


## 6. Run ICA

`FastICA` can be slow or unstable in high dimensions. The notebook therefore uses a simple preprocessing step: centering, PCA reduction only for numerical stabilization, and then FastICA.

Important: in this notebook, PCA is **not** used as a pruning method. It is only a preprocessing step before FastICA. PCA gives an orthogonal, variance-preserving coordinate system. ICA then applies a stricter criterion by rotating the whitened space toward components that are as statistically independent and non-Gaussian as possible.

The purpose of this extra strictness is to make the resulting protection vector less dependent on raw variance alone. A low-variance direction can still be protected if it belongs to a statistically separable activation source.


In [ ]:
def fit_ica_components(
    X: np.ndarray,
    n_components: int,
    random_state: int = 0,
    max_iter: int = 2000,
    tol: float = 1e-3,
):
    """
    Fit FastICA on activation matrix X and return the mixing matrix rows
    projected back to the original feature space.

    Returns:
      mixing_original : ndarray [n_components, original_dim]
          Row k is the mixing direction of ICA component k in the original
          activation space.  Concretely:
              mixing_original = ica.mixing_.T @ pca.components_
          The sign of each row is arbitrary (FastICA indeterminacy);
          downstream code uses absolute values.
      source_activations : ndarray [num_samples, n_components]
          Estimated source signal matrix S, where X_pca ≈ S @ ica.mixing_.T.
      did_converge : bool
      n_iter : int or None

    Design note – mixing vs. unmixing matrix:
      sklearn FastICA exposes two related matrices:
        - components_  (unmixing): W s.t.  S = X_pca @ W.T
        - mixing_      (mixing):   A s.t.  X_pca ≈ S @ A.T
      For computing which original dimensions *participate* in each
      component (the 'loading' in the README notation), we want the
      mixing matrix: column k of A tells how strongly component k mixes
      into each PCA-space dimension, and projecting back via
      pca.components_ gives the original-space loadings.
      The unmixing matrix (components_) answers the dual question:
      which directions to project the input onto in order to *extract*
      each component.  Both are numerically close when whitening makes
      them approximate inverses, but the mixing matrix is the correct
      choice to match the mathematical notation in the README.

    Notes:
      FastICA sometimes emits a ConvergenceWarning on LLM activations.
      This function catches the warning and returns did_converge=False
      so downstream code can skip unstable modules if configured to do so.
    """
    X = np.asarray(X, dtype=np.float32)
    X = X[np.isfinite(X).all(axis=1)]
    if X.shape[0] < max(10, n_components + 2):
        raise ValueError(f"Too few samples for ICA: {X.shape}")

    original_dim = X.shape[1]
    n_comp = min(n_components, original_dim, X.shape[0] - 1)
    X_centered = X - X.mean(axis=0, keepdims=True)

    pca = PCA(n_components=n_comp, svd_solver="randomized", random_state=random_state)
    X_pca = pca.fit_transform(X_centered)

    ica = FastICA(
        n_components=n_comp,
        whiten="unit-variance",
        max_iter=max_iter,
        tol=tol,
        algorithm="parallel",
        fun="logcosh",
        random_state=random_state,
    )

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        S = ica.fit_transform(X_pca)  # S shape: [n_samples, n_comp]

    did_converge = not any(issubclass(w.category, ConvergenceWarning) for w in caught)

    # FIX: use mixing matrix (ica.mixing_) instead of unmixing matrix
    # (ica.components_) so that row k of mixing_original represents
    # how strongly component k mixes into each original dimension –
    # i.e., the 'loading' as defined in the README.
    # ica.mixing_ shape: [n_comp, n_comp] in whitened PCA space
    # ica.mixing_.T shape: [n_comp, n_comp]  (row k = mixing dir of comp k)
    # pca.components_ shape: [n_comp, original_dim]
    # result: [n_comp, original_dim]
    mixing_original = ica.mixing_.T @ pca.components_

    return (
        mixing_original.astype(np.float32),
        S.astype(np.float32),
        did_converge,
        getattr(ica, "n_iter_", None),
    )


def component_importance_from_sources(S: np.ndarray, eps: float = 1e-8):
    """Mean absolute source activation, normalised to sum to 1."""
    imp = np.mean(np.abs(S), axis=0)
    imp = imp / (imp.sum() + eps)
    return imp.astype(np.float32)


In [ ]:
ica_results = {}

for name in tqdm(list(target_modules.keys()), desc="Fitting ICA per module"):
    try:
        mixing_in, S_in, conv_in, n_iter_in = fit_ica_components(
            X_in[name],
            cfg.n_components,
            random_state=cfg.random_seed,
            max_iter=cfg.ica_max_iter,
            tol=cfg.ica_tol,
        )
        if cfg.skip_nonconverged_ica and not conv_in:
            print(f"[WARN] Input ICA did not converge for {name}; skipping this module.")
            continue

        imp_in = component_importance_from_sources(S_in)
        result = {
            # FIX: renamed components_in → mixing_in to match mixing-matrix semantics
            "mixing_in": mixing_in,
            "importance_in": imp_in,
            "input_ica_converged": conv_in,
            "input_ica_n_iter": n_iter_in,
        }

        if cfg.use_output_ica and name in X_out:
            mixing_out, S_out, conv_out, n_iter_out = fit_ica_components(
                X_out[name],
                cfg.n_components,
                random_state=cfg.random_seed + 1,
                max_iter=cfg.ica_max_iter,
                tol=cfg.ica_tol,
            )
            if cfg.skip_nonconverged_ica and not conv_out:
                print(
                    f"[WARN] Output ICA did not converge for {name}; "
                    "using input-only score for this module."
                )
            else:
                imp_out = component_importance_from_sources(S_out)
                result["mixing_out"] = mixing_out
                result["importance_out"] = imp_out
                result["output_ica_converged"] = conv_out
                result["output_ica_n_iter"] = n_iter_out

        ica_results[name] = result
    except Exception as e:
        print("ICA failed for", name, "->", repr(e))

print("ICA successful modules:", len(ica_results))


## 7. Weight-level score and proposed pruning mask

For each linear module, the weight matrix has shape:

$$
W \in \mathbb{R}^{d_{\mathrm{out}} \times d_{\mathrm{in}}}
$$

The score matrix has the same shape as $W$.

### Input-only score

$$
\mathrm{score}_{ij}
=
\left|W_{ij}\right| \cdot p^{\mathrm{in}}_j
$$

### Input-output score

$$
\mathrm{score}_{ij}
=
\left|W_{ij}\right|
\cdot p^{\mathrm{out}}_i
\cdot p^{\mathrm{in}}_j
$$

Interpretation:

- high score: protect this weight,
- low score: candidate for additional pruning,
- already-zero weights are ignored when proposing new pruning masks.

The lowest-scoring nonzero weights are selected for the proposed additional pruning mask.


In [ ]:

def protection_vector(mixing: np.ndarray, importance: np.ndarray, eps: float = 1e-8):
    """Compute a per-dimension protection score from the mixing matrix.

    Args:
        mixing:     [n_components, dim]  – mixing matrix rows in original space.
                    Row k is the mixing direction of component k.
        importance: [n_components]      – normalised component importance weights.

    Returns:
        p: [dim]  – protection score per input or output dimension,
                    normalised so that p.mean() ≈ 1.
    """
    p = (np.abs(mixing) * importance[:, None]).sum(axis=0)  # [dim]
    p = p / (p.mean() + eps)
    return p.astype(np.float32)


def compute_score_for_module(module: nn.Linear, result: dict, use_output: bool = True):
    W = module.weight.detach().float().cpu().numpy()  # [d_out, d_in]
    p_in = protection_vector(result["mixing_in"], result["importance_in"])
    score = np.abs(W) * p_in[None, :]  # broadcast over rows

    if use_output and "mixing_out" in result and "importance_out" in result:
        p_out = protection_vector(result["mixing_out"], result["importance_out"])
        score = score * p_out[:, None]  # broadcast over columns
    return score.astype(np.float32)


def propose_extra_mask(module: nn.Linear, score: np.ndarray, extra_sparsity: float):
    """Select the n_prune lowest-scoring nonzero weights as pruning candidates.

    FIX: replaced np.partition + threshold comparison with a direct argsort
    approach.  The old threshold-based path could select too many or too few
    entries when tied scores landed exactly on the threshold boundary due to
    floating-point representation.  argsort with stable sort gives an exact,
    deterministic selection of exactly n_prune weights.
    """
    W = module.weight.detach().float().cpu().numpy()
    nonzero = W != 0
    eligible_indices = np.argwhere(nonzero)  # [n_eligible, 2]
    n_eligible = eligible_indices.shape[0]
    n_prune = int(round(extra_sparsity * n_eligible))
    mask = np.zeros_like(W, dtype=bool)
    if n_prune <= 0 or n_eligible == 0:
        return mask

    eligible_scores = score[nonzero]  # [n_eligible]
    # stable argsort: ties broken by original array order (reproducible)
    sorted_order = np.argsort(eligible_scores, kind="stable")[:n_prune]
    selected = eligible_indices[sorted_order]  # [n_prune, 2]
    mask[selected[:, 0], selected[:, 1]] = True
    return mask


def sparsity_of_weight(module: nn.Linear):
    W = module.weight.detach()
    return float((W == 0).sum().item() / W.numel())


In [ ]:

scores_dir = Path(cfg.output_dir) / "module_scores"
masks_dir = Path(cfg.output_dir) / "proposed_masks"
scores_dir.mkdir(parents=True, exist_ok=True)
masks_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []
proposed_masks = {}

for name, result in tqdm(ica_results.items(), desc="Scoring modules"):
    module = target_modules[name]
    score = compute_score_for_module(module, result, use_output=cfg.use_output_ica)
    mask = propose_extra_mask(module, score, cfg.extra_sparsity)
    proposed_masks[name] = mask

    safe_name = name.replace(".", "__")
    torch.save(torch.from_numpy(score), scores_dir / f"{safe_name}.pt")
    torch.save(torch.from_numpy(mask), masks_dir / f"{safe_name}.pt")

    W_np = module.weight.detach().cpu().numpy()
    existing = sparsity_of_weight(module)
    nonzero_count = int((W_np != 0).sum())
    extra = float(mask.sum() / max(1, nonzero_count))
    total = float(((W_np == 0) | mask).sum() / W_np.size)

    summary_rows.append({
        "module_name": name,
        "weight_shape": str(tuple(module.weight.shape)),
        "existing_sparsity": existing,
        "proposed_extra_sparsity_on_nonzero": extra,
        "proposed_total_sparsity": total,
        "score_mean": float(np.mean(score)),
        "score_std": float(np.std(score)),
        "score_min": float(np.min(score)),
        "score_max": float(np.max(score)),
        "proposed_pruned_weights": int(mask.sum()),
    })

summary = pd.DataFrame(summary_rows)
summary_path = Path(cfg.output_dir) / "summary.csv"
summary.to_csv(summary_path, index=False)
summary.head(20)


## 8. Interpret the summary

`summary.csv` reports, for each module, the current sparsity, proposed extra sparsity, estimated new total sparsity, score statistics, and the number of weights proposed for pruning.

This is still a **proposal**, not proof that the weights are useless. The primary purpose is to identify where additional redundancy may exist.


In [ ]:

summary.sort_values("proposed_pruned_weights", ascending=False).head(20)


## 9. Optional: apply the mask

By default, the notebook does not modify the model. If `cfg.apply_mask = True`, the selected weights are zeroed.

Recommended workflow: first generate only scores and masks. After applying a mask, always measure the perplexity change.


In [ ]:

def apply_masks_to_model(model, target_modules, proposed_masks):
    with torch.no_grad():
        for name, mask_np in proposed_masks.items():
            module = target_modules[name]
            mask = torch.from_numpy(mask_np).to(device=module.weight.device, dtype=torch.bool)
            module.weight[mask] = 0

if cfg.apply_mask:
    apply_masks_to_model(model, target_modules, proposed_masks)
    print("Masks applied.")
else:
    print("cfg.apply_mask is False; model was not modified.")


## 10. Simple perplexity check

The perplexity function is a quick sanity check. A rigorous evaluation should use a separate validation set and multiple downstream benchmarks.


In [ ]:

def compute_ppl(
    model,
    tokenizer,
    texts: List[str],
    seq_len: int = 256,
    max_eval_samples: int = 16,
):
    """Compute perplexity on the provided texts.

    For a meaningful perplexity estimate, pass a held-out set that was
    not used for calibration or ICA fitting.  Use eval_texts (loaded
    from the validation split) rather than the calibration texts.
    """
    losses = []
    model.eval()
    with torch.no_grad():
        for t in tqdm(texts[:max_eval_samples], desc="PPL eval"):
            batch = tokenizer(
                [t],
                padding="max_length",
                truncation=True,
                max_length=seq_len,
                return_tensors="pt",
            )
            batch = {k: v.to(model.device) for k, v in batch.items()}
            labels = batch["input_ids"].clone()
            labels[batch["attention_mask"] == 0] = -100
            out = model(**batch, labels=labels)
            losses.append(float(out.loss.detach().cpu()))
    mean_loss = float(np.mean(losses))
    return math.exp(mean_loss), mean_loss


# FIX: evaluate on held-out eval_texts, NOT on calibration texts.
# Evaluating on calibration data yields an optimistically biased perplexity
# because the ICA and score were derived from those same activations.
ppl, loss = compute_ppl(
    model,
    tokenizer,
    eval_texts,           # <-- held-out split
    seq_len=cfg.seq_len,
    max_eval_samples=min(16, len(eval_texts)),
)
print("Perplexity (held-out eval set):", ppl, "  loss:", loss)


## 11. Save outputs

The notebook saves the configuration, module-level score matrices, proposed pruning masks, the summary CSV, and optionally the pruned model.


In [ ]:

with open(Path(cfg.output_dir) / "config.json", "w", encoding="utf-8") as f:
    json.dump(asdict(cfg), f, indent=2, ensure_ascii=False)

with open(Path(cfg.output_dir) / "run_log.txt", "w", encoding="utf-8") as f:
    f.write("ICA-based second-pass pruning scan\n")
    f.write(f"model: {cfg.model_name_or_path}\n")
    f.write(f"modules scored: {len(ica_results)}\n")
    f.write(f"perplexity_after_optional_mask: {ppl}\n")

if cfg.apply_mask:
    save_dir = Path(cfg.output_dir) / "pruned_model"
    save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print("Saved pruned model to", save_dir)

print("Saved outputs to", cfg.output_dir)
print("Summary:", summary_path)


## 12. Suggested experiments and limitations

Suggested experiments:

1. Run the scan on both an unpruned baseline model and an already-pruned model.
2. Compare the ICA-proposed extra mask against an additional magnitude-based pruning pass.
3. Compare the ICA-proposed extra mask against an SVD/orthogonal-basis second pass.
4. Compare perplexity degradation at the same extra sparsity level.
5. Test MLP modules and attention projection modules separately.
6. Compare input-only scoring against input-output scoring.

Limitations:

- ICA components are not guaranteed to be human-interpretable concepts.
- ICA is stricter than orthogonal basis search, but stricter does not automatically mean causal. It means the components satisfy a stronger statistical separation criterion.
- The score is a local, layer-wise approximation.
- The component-to-weight mapping is not a complete causal proof.
- Cross-layer residual-stream effects are only observed indirectly through calibration activations.
- Rigorous evaluation requires downstream benchmarks and possibly recovery fine-tuning.

Core flow:

$$
\mathrm{activations}
\;\longrightarrow\;
\mathrm{ICA\ components}
\;\longrightarrow\;
\mathrm{weight\ protection\ map}
\;\longrightarrow\;
\mathrm{proposed\ extra\ pruning\ mask}
$$

This does not replace Wanda or SparseGPT. It is intended as a second-pass diagnostic layer that tries to find additional pruning opportunities after conventional pruning methods have already removed the obvious redundancy. The expected benefit over an orthogonal second pass is that ICA can protect or expose source-like activation structure that is not cleanly aligned with singular vectors.
